# Sumativa 3 — Parte 1: imputación y manejo de datos faltantes

Esta sección utiliza explícitamente los resultados de S1 y S2. S1 identificó 582 filas incompletas (13,7 %) y siete variables con faltantes; `glucose` fue la más afectada (9,2 %). También observó que los 29 faltantes de `cigsPerDay` pertenecen exclusivamente a fumadores. S2 validó como estables las relaciones PAS–PAD, edad–PAS e IMC–PAD y recomendó ajustar cualquier imputación solo con entrenamiento cuando se evalúe un modelo predictivo.

S1 no aplicó una prueba formal para clasificar globalmente el mecanismo MCAR/MAR/MNAR. Por ello no se afirma que MAR haya sido demostrado: se adopta **MAR como supuesto operativo**, pues la ausencia de `cigsPerDay` depende de una variable observada (`currentSmoker`) y los demás faltantes pueden modelarse con covariables observadas. La comparación de tres estrategias funciona como análisis de sensibilidad frente a este supuesto.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from utils_s3 import (
    cargar_datos, resumen_faltantes, imputar_simple, imputar_regresion,
    comparar_distribuciones, comparar_correlaciones
)

BASE = Path('.')
DATOS_SALIDA = BASE / 'datos'
TABLAS = BASE / 'tablas'
FIGURAS = BASE / 'figuras'
for ruta in (DATOS_SALIDA, TABLAS, FIGURAS):
    ruta.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')

## 1.1 Resultados de faltantes heredados de S1

No se repite el análisis exploratorio completo; se recuperan los conteos necesarios para documentar las decisiones de S3.

In [ ]:
original = cargar_datos()
faltantes = resumen_faltantes(original)
faltantes.to_csv(TABLAS / 'p1_faltantes_s1.csv')
display(faltantes.round(3))
print(f'Filas con al menos un faltante: {original.isna().any(axis=1).sum()} ({100*original.isna().any(axis=1).mean():.1f} %)')

ax = faltantes.sort_values('porcentaje_faltantes')['porcentaje_faltantes'].plot.barh(
    figsize=(7, 4), color='#2A6F97'
)
ax.set(xlabel='Porcentaje de valores faltantes', ylabel='', title='Faltantes documentados en S1')
plt.tight_layout()
plt.savefig(FIGURAS / 'p1_fig1_faltantes.png', dpi=180)
plt.show()

## 1.2 Imputación mediante regresión lineal múltiple

`imputar_regresion()` está implementada en `utils_s3.py`. Para cada variable numérica incompleta se ajusta una regresión con observaciones completas y 3–5 predictores. Se priorizan las correlaciones estables de S2 y nunca se usa `TenYearCHD` como predictor de imputación. Las predicciones se restringen al rango observado y la mediana actúa únicamente como respaldo si una fila carece de predictores.

El desempeño se estima en una validación 80/20 dentro de los casos completos. Jarque–Bera, Breusch–Pagan, VIF y distancia de Cook permiten evaluar los supuestos; una imputación puede ser útil para preservar tamaño muestral aunque su R² sea modesto, pero esa limitación debe declararse.

In [ ]:
regresion, estado_regresion, diagnosticos, conteos = imputar_regresion(
    original, devolver_detalle=True
)
diagnosticos.to_csv(TABLAS / 'p1_diagnosticos_imputacion.csv', index=False)
conteos.to_csv(TABLAS / 'p1_conteos_imputados.csv', index=False)
display(conteos.fillna('').style.format(precision=0))
display(diagnosticos.style.format({
    'R2_validacion': '{:.3f}', 'RMSE_validacion': '{:.3f}',
    'MAE_validacion': '{:.3f}', 'JB_p_residuos': '{:.3g}',
    'BP_p_heterocedasticidad': '{:.3g}', 'VIF_max': '{:.2f}',
    'Cook_max': '{:.3f}'
}))
assert regresion.isna().sum().sum() == 0

La intensidad de tabaquismo presenta el mejor desempeño de imputación porque `currentSmoker` aporta información directa. Para colesterol, IMC, frecuencia cardiaca y glucosa el error es mayor; además, la normalidad y homocedasticidad pueden incumplirse por la asimetría clínica ya observada en S1/S2. Por ello los valores imputados no deben interpretarse como mediciones individuales exactas: su finalidad es conservar la estructura del conjunto para el modelamiento y se exige contrastar las conclusiones con eliminación e imputación simple.

## 1.3 Construcción y comparación de las tres estrategias

1. **Eliminación completa:** conserva solo filas sin faltantes.  
2. **Imputación simple:** mediana para numéricas y moda para categóricas.  
3. **Imputación por regresión:** regresiones múltiples para numéricas y moda para categóricas.

In [ ]:
eliminacion = original.dropna().copy()
simple = imputar_simple(original)
datasets = {'eliminacion': eliminacion, 'simple': simple, 'regresion': regresion}

for nombre, df in datasets.items():
    df.to_csv(DATOS_SALIDA / f'framingham_{nombre}.csv', index=False)

tamanos = pd.DataFrame([
    {
        'estrategia': nombre, 'n': len(df),
        'filas_perdidas': len(original) - len(df),
        'eventos_CHD': int(df['TenYearCHD'].sum()),
        'tasa_CHD': df['TenYearCHD'].mean()
    }
    for nombre, df in datasets.items()
])
distribuciones = comparar_distribuciones(original, datasets)
correlaciones = comparar_correlaciones(datasets)
tamanos.to_csv(TABLAS / 'p1_comparacion_tamanos.csv', index=False)
distribuciones.to_csv(TABLAS / 'p1_comparacion_distribuciones.csv', index=False)
correlaciones.to_csv(TABLAS / 'p1_preservacion_correlaciones.csv', index=False)
display(tamanos.style.format({'tasa_CHD': '{:.3%}'}))
display(distribuciones.round(3))
display(correlaciones.round(4))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
variables = ['cigsPerDay', 'totChol', 'BMI', 'heartRate', 'glucose']
colores = {'eliminacion': '#6C757D', 'simple': '#E09F3E', 'regresion': '#2A6F97'}
for ax, variable in zip(axes.ravel(), variables):
    for nombre, df in datasets.items():
        sns.kdeplot(df[variable], ax=ax, label=nombre, color=colores[nombre], warn_singular=False)
    ax.set_title(variable)
axes.ravel()[-1].axis('off')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right')
fig.suptitle('Distribuciones bajo las tres estrategias de faltantes')
fig.tight_layout()
plt.savefig(FIGURAS / 'p1_fig2_distribuciones_imputacion.png', dpi=180, bbox_inches='tight')
plt.show()

## 1.4 Decisión para las fases posteriores

Se selecciona **imputación por regresión** como estrategia principal porque conserva los 4.240 participantes y usa relaciones multivariadas justificadas por S1/S2. La eliminación completa pierde 582 registros (13,7 %) y 87 eventos de CHD, lo que reduce información de la clase minoritaria. La imputación simple también conserva el tamaño, pero comprime artificialmente la variabilidad alrededor de medianas/modas.

La decisión no implica que la regresión reconstruya con precisión cada medición: los R² modestos y los incumplimientos de supuestos obligan a mantener la comparación de sensibilidad de la Parte 4. Los tres CSV quedan guardados en `S3/datos/` para que el trabajo grupal utilice exactamente las mismas versiones.